[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [scikit-learn, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)

# Pipelines &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler

def station_network(n=1200, seed=2025):
    """A made-up network of weather stations in Norway, one row each, with three things to predict."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    terrain = np.where(coast_km < 3, "coastal",
              np.where(elevation > 700, "mountain",
              np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), "fjord", "inland")))
    owner = rng.choice(["met", "road", "airport", "private"], n, p=[0.4, 0.3, 0.1, 0.2])
    installed = rng.integers(1950, 2025, n)
    shift = pd.Series(terrain).map({"coastal": 0.8, "fjord": 0.3, "inland": 0.0, "mountain": -0.5}).to_numpy()
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    june_min = 8.5 - 0.35 * (latitude - 58) - 0.0075 * elevation - 0.02 * coast_km + rng.normal(0, 1.5, n)
    table = pd.DataFrame({
        "station": [f"ST{i:04d}" for i in range(1, n + 1)],
        "latitude": latitude.round(2), "longitude": longitude.round(2),
        "elevation_m": elevation.round(0).astype(int), "coast_km": coast_km.round(1),
        "terrain": terrain, "owner": owner, "installed": installed,
        "jan_mean": jan_mean.round(1),
        "snow_on_may_1": may_mean < 2.5,
        "june_frost": june_min < -0.8,
    })
    return table


network = station_network()
FEATURES = ["latitude", "longitude", "elevation_m", "coast_km", "installed"]

print("scikit-learn", sklearn.__version__, "|", len(network), "stations")

scikit-learn 1.6.1 | 1200 stations


**1.** Make a pipeline of `StandardScaler` and `LogisticRegression` that predicts `snow_on_may_1` from `latitude` and `elevation_m` alone, fit it on a split with `test_size=0.25`, `random_state=0` and `stratify`, and print its accuracy on the held-back stations.

In [2]:
two = network[["latitude", "elevation_m"]]
snow = network["snow_on_may_1"]
X_train, X_test, y_train, y_test = train_test_split(two, snow, test_size=0.25, random_state=0, stratify=snow)

small = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_train, y_train)
print(round(small.score(X_test, y_test), 3))

0.81


`fit` returns the pipeline, so it can be made and fitted in one line.

**2.** For the pipeline from task 1, print the names of its steps and the means its scaler learned.

In [3]:
print(list(small.named_steps))
print(small.named_steps["standardscaler"].mean_.round(2))

['standardscaler', 'logisticregression']
[ 64.52 290.69]


The scaler learned two means, one for each column, from the 900 training stations.

**3.** Change the pipeline's `C` to 0.01 with `set_params`, fit it again, and print its accuracy on the held-back stations.

In [4]:
small.set_params(logisticregression__C=0.01)
small.fit(X_train, y_train)

print(round(small.score(X_test, y_test), 3))

0.783


`set_params` changes the setting, and the model learns nothing new until `fit` runs again.

**4.** Cross-validate a pipeline of `StandardScaler` and `LogisticRegression` on all five `FEATURES` and `snow_on_may_1` with `cv=10`, and print the ten scores and their mean, rounded to three places.

In [5]:
scores = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression()),
                         network[FEATURES], network["snow_on_may_1"], cv=10)

print(scores.round(3))
print(scores.mean().round(3))

[0.783 0.833 0.883 0.833 0.9   0.825 0.892 0.817 0.825 0.883]
0.848


Each of the ten folds fitted its own scaler and model on the other nine tenths of the stations.

**5.** Repeat the noise demonstration with `k=5` columns instead of 20, both outside and inside the folds, and print the two mean scores.

In [6]:
noise = pd.DataFrame(np.random.default_rng(0).normal(size=(200, 2000)))
coin = np.random.default_rng(1).integers(0, 2, 200)

chosen_once = SelectKBest(f_classif, k=5).fit(noise, coin)
outside = cross_val_score(LogisticRegression(), chosen_once.transform(noise), coin, cv=5)
inside = cross_val_score(make_pipeline(SelectKBest(f_classif, k=5), LogisticRegression()), noise, coin, cv=5)

print("outside:", outside.mean().round(3))
print("inside: ", inside.mean().round(3))
print(inside.round(3))

outside: 0.675
inside:  0.57
[0.475 0.475 0.55  0.675 0.675]


The first two lines rebuild the noise as the worked example did, since this notebook runs on its own. Five columns chosen outside the folds scored 0.675, against 0.57 inside them. With fewer columns chosen there is less luck to find, so the gap is smaller than with 20, and the score from outside the folds is still the one that measures that luck.

---

&#8592; **Back to:** [Pipelines](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/05-pipelines.ipynb)  &nbsp;&middot;&nbsp;  [scikit-learn, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)
